# 04 — Label Construction

**Goal:** For every hourly row, compute how much time was *actually* left until that cycle
crossed the dry threshold, then bucket it into an urgency class:

| Time remaining | Class |
|---|---|
| < 24h | `Urgent` |
| 24–48h | `Soon` |
| > 48h | `Not Urgent` |

This turns each depletion cycle (one physical event) into a full time series of labeled
examples — the key move that gets ~150–400+ training rows out of only 10–17 real cycles.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

PREPROCESSED_DIR = Path("../data/preprocessed")

features_df = pd.read_csv(PREPROCESSED_DIR / "features_only.csv", parse_dates=["timestamp"])
print(features_df.shape)
features_df.head()

(82, 9)


,timestamp,cycle_id,condition,temperature_C,humidity_pct,soil_moisture_pct,moisture_trend,light_lux,hour_of_day
0,2026-07-01 12:00:00,indoor_cycle01,indoor,27.65,47.72,79.65,-7.46,155.5,12
1,2026-07-01 13:00:00,indoor_cycle01,indoor,26.84,53.37,77.56,-6.83,170.5,13
2,2026-07-01 14:00:00,indoor_cycle01,indoor,28.18,52.27,75.50,-6.51,186.0,14
3,2026-07-01 15:00:00,indoor_cycle01,indoor,26.73,53.51,72.95,-6.70,155.3,15
4,2026-07-01 16:00:00,indoor_cycle01,indoor,26.39,53.02,70.03,-7.53,158.0,16


## Compute time-remaining per row

For each cycle, the last row's timestamp is the threshold-crossing time (`T`). Every row's label is based on `T - t`.

In [2]:
def add_time_remaining(df):
    df = df.sort_values("timestamp").copy()
    cycle_end_time = df["timestamp"].iloc[-1]
    df["hours_remaining"] = (cycle_end_time - df["timestamp"]).dt.total_seconds() / 3600.0
    return df

labeled_frames = []
for cycle_id, group in features_df.groupby("cycle_id"):
    labeled_frames.append(add_time_remaining(group))

labeled_df = pd.concat(labeled_frames, ignore_index=True)
labeled_df[["cycle_id", "timestamp", "hours_remaining"]].head(10)

,cycle_id,timestamp,hours_remaining
0,indoor_cycle01,2026-07-01 12:00:00,24.0
1,indoor_cycle01,2026-07-01 13:00:00,23.0
2,indoor_cycle01,2026-07-01 14:00:00,22.0
3,indoor_cycle01,2026-07-01 15:00:00,21.0
4,indoor_cycle01,2026-07-01 16:00:00,20.0
5,indoor_cycle01,2026-07-01 17:00:00,19.0
6,indoor_cycle01,2026-07-01 18:00:00,18.0
7,indoor_cycle01,2026-07-01 19:00:00,17.0
8,indoor_cycle01,2026-07-01 20:00:00,16.0
9,indoor_cycle01,2026-07-01 21:00:00,15.0


## Map time-remaining to urgency bucket

In [3]:
def urgency_bucket(hours_remaining):
    if hours_remaining < 24:
        return "Urgent"       # <24h
    elif hours_remaining < 48:
        return "Soon"         # 24-48h
    else:
        return "Not_Urgent"   # >48h

labeled_df["urgency_class"] = labeled_df["hours_remaining"].apply(urgency_bucket)
print(labeled_df["urgency_class"].value_counts())

urgency_class
Urgent    80
Soon       2
Name: count, dtype: int64


## Quick sanity check

The very last row of every cycle should have `hours_remaining` == 0 and be labeled `Urgent`.

In [4]:
last_rows = labeled_df.groupby("cycle_id").tail(1)
print(last_rows[["cycle_id", "hours_remaining", "urgency_class"]])
assert (last_rows["hours_remaining"] == 0).all(), "Last row of a cycle should have 0 hours remaining"
print("\nSanity check passed: every cycle's final row is 0h remaining.")

           cycle_id  hours_remaining urgency_class
24   indoor_cycle01              0.0        Urgent
49   indoor_cycle02              0.0        Urgent
65  outdoor_cycle01              0.0        Urgent
81  outdoor_cycle02              0.0        Urgent

Sanity check passed: every cycle's final row is 0h remaining.


## Save the final labeled dataset

In [5]:
FEATURE_COLS = ["temperature_C", "humidity_pct", "soil_moisture_pct", "moisture_trend", "light_lux", "hour_of_day"]

final_cols = ["cycle_id", "condition", "timestamp"] + FEATURE_COLS + ["hours_remaining", "urgency_class"]
labeled_df = labeled_df[final_cols]

labeled_df.to_csv(PREPROCESSED_DIR / "labeled_dataset.csv", index=False)
print(f"Saved labeled_dataset.csv — {len(labeled_df)} rows, {labeled_df['cycle_id'].nunique()} cycles")
labeled_df.head()

Saved labeled_dataset.csv — 82 rows, 4 cycles


,cycle_id,condition,timestamp,temperature_C,humidity_pct,soil_moisture_pct,moisture_trend,light_lux,hour_of_day,hours_remaining,urgency_class
0,indoor_cycle01,indoor,2026-07-01 12:00:00,27.65,47.72,79.65,-7.46,155.5,12,24.0,Soon
1,indoor_cycle01,indoor,2026-07-01 13:00:00,26.84,53.37,77.56,-6.83,170.5,13,23.0,Urgent
2,indoor_cycle01,indoor,2026-07-01 14:00:00,28.18,52.27,75.50,-6.51,186.0,14,22.0,Urgent
3,indoor_cycle01,indoor,2026-07-01 15:00:00,26.73,53.51,72.95,-6.70,155.3,15,21.0,Urgent
4,indoor_cycle01,indoor,2026-07-01 16:00:00,26.39,53.02,70.03,-7.53,158.0,16,20.0,Urgent


**Next step:** `05_exploratory_data_analysis.ipynb` — check class balance and feature distributions before trusting this dataset with a model.